# ToolRL Reproduction Notebook

Provisions a Chameleon Cloud GPU instance, sets up Docker, and runs one training case end-to-end.

**Target:** GRPO [RL algorithm that doesn't need a separate critic model] Cold Start [trained straight from the base model, no supervised fine-tuning first], Qwen2.5-1.5B
**Paper result (API-Bank [the paper's own 597-question tool-call benchmark]):** 63.15% | **Our result:** 58.96%
**Wall time:** ~3 hours on 4x H100

**Before running:**
- Chameleon Cloud account with an active allocation
- SSH keypair registered at chi.uc.chameleoncloud.org
- `pip install python-chi`

## 0. Configuration

In [ ]:
SITE         = "CHI@UC"
PROJECT_NAME = "CHI-XXXXXX"       # your allocation number, e.g. CHI-251234
KEYPAIR_NAME = "my-chameleon-key" # as it appears in the dashboard at chi.uc.chameleoncloud.org
SSH_KEY_PATH = "~/.ssh/id_rsa"
FLAVOR       = "m1.xlarge.gpu"    # 4x H100 80GB VM (KVM) at CHI@UC -- confirm exact name in dashboard: Compute > Instances > Launch > Flavor
LEASE_HOURS  = 6
REPO_URL     = "https://github.com/Mario928/toolrl-verl-reproduction"

## 1. Provision the Instance

In [ ]:
import chi, chi.lease, chi.server, datetime

chi.use_site(SITE)
chi.set("project_name", PROJECT_NAME)

start = datetime.datetime.utcnow() + datetime.timedelta(minutes=1)
end   = start + datetime.timedelta(hours=LEASE_HOURS)

# KVM@UC uses floating-instance reservations (not node reservations like bare metal CHI@UC)
lease = chi.lease.create_lease(
    "toolrl-reproduce",
    reservations=[chi.lease.get_instance_reservation(min_count=1, max_count=1, flavor_id=chi.server.get_flavor(FLAVOR).id)],
    start_date=start,
    end_date=end,
)
chi.lease.wait_for_active(lease["id"])
print("Lease active:", lease["id"])

In [ ]:
reservation_id = chi.lease.get_instance_reservation(lease["id"])
server = chi.server.create_server(
    "toolrl-node",
    reservation_id=reservation_id,
    image_name="CC-Ubuntu22.04",
    key_name=KEYPAIR_NAME,
    flavor_name=FLAVOR,
)
chi.server.wait_for_active(server.id)
floating_ip = chi.server.associate_floating_ip(server.id)
print("Floating IP:", floating_ip)

## 2. Connect via SSH

In [ ]:
import chi.ssh, os

node = chi.ssh.Remote(floating_ip, username="cc", key_filename=os.path.expanduser(SSH_KEY_PATH))
stdout, _ = node.execute("uname -a")
print(stdout)

## 3. Install Docker and nvidia-container-toolkit

In [ ]:
node.execute("sudo apt-get update -q")
node.execute("sudo apt-get install -y -q ca-certificates curl gnupg lsb-release")
node.execute("curl -fsSL https://download.docker.com/linux/ubuntu/gpg | sudo gpg --dearmor -o /usr/share/keyrings/docker-archive-keyring.gpg")
node.execute('echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/docker-archive-keyring.gpg] https://download.docker.com/linux/ubuntu $(lsb_release -cs) stable" | sudo tee /etc/apt/sources.list.d/docker.list > /dev/null')
node.execute("sudo apt-get update -q")
node.execute("sudo apt-get install -y -q docker-ce docker-ce-cli containerd.io docker-compose-plugin")
node.execute("sudo usermod -aG docker cc")
print("Docker installed.")

In [ ]:
node.execute("curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg")
node.execute("curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list | sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list")
node.execute("sudo apt-get update -q")
node.execute("sudo apt-get install -y -q nvidia-container-toolkit")
node.execute("sudo nvidia-ctk runtime configure --runtime=docker")
node.execute("sudo systemctl restart docker")
print("nvidia-container-toolkit installed.")

In [ ]:
stdout, _ = node.execute("sudo docker run --rm --gpus all nvidia/cuda:12.1.0-base-ubuntu22.04 nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print(stdout)

## 4. Clone Repo and Create Volumes

In [ ]:
node.execute(f"git clone {REPO_URL} /home/cc/toolrl")
for vol in ["math-reasoning-rl_models", "math-reasoning-rl_hf_cache", "math-reasoning-rl_mlflow_data", "math-reasoning-rl_datasets"]:
    node.execute(f"sudo docker volume create {vol}")
print("Done.")

## 5. Build the Docker Image

Takes ~15-20 minutes the first time.

In [ ]:
stdout, _ = node.execute("cd /home/cc/toolrl && sudo docker build -t toolrl-verl:latest . 2>&1 | tail -5")
print(stdout)

## 6. Start Containers

In [ ]:
node.execute("cd /home/cc/toolrl && sudo docker compose up -d")
stdout, _ = node.execute("sudo docker ps --format 'table {{.Names}}\t{{.Status}}'")
print(stdout)

## 7. Prepare the Dataset

Converts the raw JSON files already in the repo into the parquet format the trainer expects. Must run before training or the trainer crashes at step 1 with `KeyError: 'reward_model'`.

In [ ]:
## 8. Download the Base Model

## 7. Download the Base Model

In [ ]:
## 9. Run Training

GRPO [RL algorithm, no critic model] cold start [no supervised fine-tuning beforehand], 15 epochs, 4 GPUs. Final checkpoint at `global_step_90`.
To run a different cell, change `BASE_MODEL`, `EXPERIMENT_NAME`, and the reward flags.

## 8. Run Training

GRPO [RL algorithm, no critic model] cold start [no supervised fine-tuning beforehand], 15 epochs, 4 GPUs. Final checkpoint at `global_step_90`.
To run a different cell, change `BASE_MODEL`, `EXPERIMENT_NAME`, and the reward flags.

In [ ]:
node.execute("""
sudo docker exec -d verl bash -c '
    export CUDA_VISIBLE_DEVICES=0,1,2,3
    export N_GPUS=4
    export ROLLOUT_TP_SIZE=1
    export VLLM_ATTENTION_BACKEND=XFORMERS
    export WITHLENGTH=0 REFINEDREWARD=0 COARSEREWARD=0 STRICTMATCH=0
    export CORRECTMAX1=0 MAX1STEP30MAX3=0 SCHEDULEREWARD=0 SCHEDULELENGTH=0
    export DATA_DIR="./dataset/rlla_4k"
    export BASE_MODEL="Qwen/Qwen2.5-1.5B-Instruct"
    export EXPERIMENT_NAME="/app/models/toolrl-grpo-cold-qwen-1.5b"
    cd /workspace && bash ./examples/grpo_trainer/run_grpo.sh > /tmp/train.log 2>&1
'
""")
print("Training started. Run the next cell to check progress.")

In [ ]:
## 10. Evaluate on API-Bank

Run after training finishes.

CHECKPOINT = "/app/models/toolrl-grpo-cold-qwen-1.5b/actor/global_step_90"

node.execute(f"""
sudo docker exec verl bash -c '
    cd /workspace/benchmarks/API-Bank &&
    WORLD_SIZE=4 python3 generate_batch.py --model_paths {CHECKPOINT} > /tmp/apibank_gen.log 2>&1
'
""")
print("Generation done.")

In [ ]:
stdout, _ = node.execute(f"""
sudo docker exec verl bash -c '
    cd /workspace/benchmarks/API-Bank &&
    python3 evaluate.py --model_paths {CHECKPOINT}
'
""")
print(stdout)

In [ ]:
stdout, _ = node.execute(f"""
sudo docker exec verl bash -c '
    cd /workspace/benchmarks/API-Bank &&
    python3 evaluate.py --model_path {CHECKPOINT}
'
""")
print(stdout)

## 11. Cleanup

## 10. Cleanup

In [ ]:
chi.lease.delete_lease(lease["id"])
print("Lease deleted.")